# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided exploration of the FAIR^2 Mass Spectrometry dataset using the `mlcroissant` library, following best practices to reference all entities by their `@id` as per the Croissant schema.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and contains multiple record sets, fields, and columns.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")
print("\nCroissant Dataset Identifier: ", metadata.identifier)
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. All references use Croissant `@id` as mandated.

* Display all record sets and fields, including their `@id`.
* Use these IDs in all subsequent data analysis steps.

In [ ]:
print("--- Available Record Sets ---")
all_record_sets = list(metadata.record_sets)
for rs in all_record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    # List all field names and their @ids
    field_info = [(f"  Field: {field.name}  @id: {field.id}  (type: {field.data_type})", field) for field in rs.fields]
    if field_info:
        for label, field in field_info:
            print(label)
    else:
        print("  [No fields listed]")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities (record set and field) are referred to by their `@id`.

In [ ]:
# Select record set(s) by @id for extraction
# List all record set @ids from previous overview

record_set_ids = [rs.id for rs in metadata.record_sets]  # Example: ['https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034']
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Preview the columns of the first record set
if len(record_set_ids) > 0 and not dataframes[record_set_ids[0]].empty:
    print(f"Columns for record set {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())
else:
    print("No dataframes extracted or dataframe is empty.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, using only field `@id` references.

In [ ]:
# For demonstration, automatically pick the first numeric field in the first non-empty record set
record_set_id = None
numeric_field_id = None
group_field_id = None

for rs in metadata.record_sets:
    df = dataframes.get(rs.id, pd.DataFrame())
    if not df.empty:
        # Find all numeric fields
        numeric_fields = [f for f in rs.fields if f.data_type in ('Float', 'Integer', 'Number') and f.id in df.columns]
        if numeric_fields:
            # Use first numeric field
            numeric_field = numeric_fields[0]
            numeric_field_id = numeric_field.id
            record_set_id = rs.id
            # Try to find a field to group by (string field)
            group_fields = [f for f in rs.fields if f.data_type in ('Text',) and f.id in df.columns]
            if group_fields:
                group_field = group_fields[0]
                group_field_id = group_field.id
            break

if record_set_id is None or numeric_field_id is None:
    print("No suitable numeric field in data to analyze.")
else:
    df = dataframes[record_set_id]
    # The field id may contain colons, which are preserved as column names
    # Ensure correct selection
    print(f"Selected record set: {record_set_id}\nNumeric field: {numeric_field_id}\nGroup field: {group_field_id}")

    threshold = df[numeric_field_id].dropna().quantile(0.75) # Use 75th percentile as example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between numeric fields, referencing all fields by their `@id`.

* Histogram of the selected numeric field
* Boxplot grouped by the group field (if available)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- Loaded FAIR^2 Croissant schema dataset, referenced all entities by their `@id`.
- Explored available record sets, fields, and automatically extracted a DataFrame.
- Performed basic EDA and visualized distributions for a numeric field referenced by its `@id`.

All operations in this notebook are traceable and reproducible by the schema and `@id` field referencing, ensuring compliance and reusability for further FAIR and RAI ML analyses.